# Hyperparameter Finetuning

Systematically tests annotation pipeline hyperparameters on a fixed sample pool.

Disclaimer: Code was largely generated with the help of Claude Sonnet 3.5 (Anthropic, 2026 February - May). Prompts, code tweaks and verification by me.

---

## Design

- A **fixed pool** of sample IDs lives in `sandbox/sandbox_ids.json`.
  Every experiment draws from this pool so results are comparable across runs.
- The sandbox shares all storage with the main pipeline:
  `annotations/`, `labels/`, and `audio/` are the same directories.
  If a sample is already annotated, its cached data is reused directly.
  If not, it is annotated and the files land exactly where the main pipeline
  would have put them.
- PSST raw output is read from `annotations/{sample_id}.json` (the main
  pipeline cache). PSST is never re-run — it has no tunable parameters.
- Only Wav2ToBI peak detection and F0 classification are re-run per sweep,
  since those are the steps with tunable parameters.
- Audio is streamed on demand and optionally kept (`KEEP_AUDIO` toggle).

---

## Experiments in this notebook

| Section | Experiment | Parameter swept |
|---|---|---|
| 5 | Alignment tolerance | `ALIGNMENT_TOLERANCE_S` |
| 5 | Wav2ToBI threshold multiplier | `W2T_THRESHOLD_MULT` |
| 5 | F0 window size | `F0_WINDOW_S` |
| 5 | F0 slope threshold | `F0_SLOPE_HZ` |

---

## Output structure

```
sandbox/
├── sandbox_ids.json          ← fixed pool of sample IDs (written once)
└── results/
    └── {experiment}_{timestamp}.json   ← one file per experiment run

annotations/                  ← shared with main pipeline (PSST cache lives here)
labels/                       ← shared with main pipeline
audio/                        ← shared with main pipeline
```

## Results per testing
- `DEFAULT_ALIGNMENT_TOLERANCE_S = 0.25`
-->
`DEFAULT_ALIGNMENT_TOLERANCE_S = 0.15` (captures over 95%)

- `DEFAULT_W2T_THRESHOLD_MULT    = 1.5`
--->
`DEFAULT_W2T_THRESHOLD_MULT    = 1.0` (92.5%, equal to 0.75, more justifiable since it's a higher threshold requirement)

- `DEFAULT_F0_SLOPE_HZ           = 10.0`
--->
`DEFAULT_F0_SLOPE_HZ           = 2.0`
(F0 slope requirement captured 18% above random chance, making this the best candidate to match local F0 direction at prosodic boundaries to human-annotated results. roughly approximates intonation types. helps for silver standard.)
- `DEFAULT_F0_WINDOW_S           = 0.15`
--->
`DEFAULT_F0_WINDOW_S           = 0.20` (less fails to capture pattern, more captures artifacts)

---
## Setup

Edit this cell before running any experiment.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — CONFIGURATION                                      ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Paths ─────────────────────────────────────────────────────────
# These should match exactly what is set in 03_Annotation_Pipeline_v2.
DRIVE_ROOT   = "/content/drive/MyDrive/Capstone"
SANDBOX_DIR = SANDBOX_ROOT = f"{DRIVE_ROOT}/sandbox"


ANNOT_CACHE  = f"{DRIVE_ROOT}/annotations"    # shared with main pipeline
LABELS_ROOT  = f"{DRIVE_ROOT}/labels"         # shared with main pipeline
AUDIO_CACHE  = f"{DRIVE_ROOT}/audio"          # shared with main pipeline
WAV2TOBI_DIR = f"{DRIVE_ROOT}/models/wav2tobi"

RESULTS_DIR  = f"{SANDBOX_ROOT}/results"
PSST_CACHE   = f"{SANDBOX_ROOT}/psst_cache"   # cached PSST outputs (never re-run)

# Sandbox-specific: only the pool file and results live here

# model checkpoints
PSST_CKPT     = "NathanRoll/psst-medium-en"
WAV2TOBI_CKPT = "ReginaZ/Wav2ToBI-PB-Fuzzy"

# ── Pool configuration ────────────────────────────────────────────
# POOL_SIZE: total IDs to draw when first building the pool.
# N_EXPERIMENT: how many of those IDs each experiment uses (≤ POOL_SIZE).
POOL_SIZE       = 50
N_EXPERIMENT    = 50
LIBRITTS_SPLIT  = "train.clean.100"
RANDOM_SEED     = 42

# ── Manual ID override ────────────────────────────────────────────
# For F0 experiments you may want to specify utterances you can listen
# to and verify by ear.
# USE_MANUAL_IDS = True, then either:
#   (a) list IDs directly in MANUAL_IDS, or
#   (b) point MANUAL_IDS_FROM at a previous results JSON to reuse its IDs
USE_MANUAL_IDS  = False
MANUAL_IDS      = []       # e.g. ["1263_141777_000037_000001"]
MANUAL_IDS_FROM = ""       # e.g. f"{RESULTS_DIR}/alignment_20250101_120000.json"

# ── Audio ─────────────────────────────────────────────────────────
KEEP_AUDIO = True   # True = keep .wav in audio/ after processing

# ── Default hyperparameters (baselines for comparison) ────────────
DEFAULT_ALIGNMENT_TOLERANCE_S = 0.25   # max seconds from peak to word end
DEFAULT_W2T_THRESHOLD_MULT    = 1.5    # "is there a boundary?" (SDs above mean)
DEFAULT_W2T_HIGH_MULT         = 2.5    # "how strong is the boundary?" (SDs above mean)
# ----
DEFAULT_F0_WINDOW_S           = 0.15   # f0 window in seconds
DEFAULT_F0_SLOPE_HZ           = 10.0   # f0 slope to be considered significant
DEFAULT_CONSENSUS_WINDOW      = 1      # ±N words for PSST/W2T agreement

print("✓ Configuration loaded.")
print(f"  Drive root:   {DRIVE_ROOT}")
print(f"  Sandbox dir:  {SANDBOX_DIR}")
print(f"  Annotations:  {ANNOT_CACHE}  (shared with main pipeline)")
print(f"  Labels:       {LABELS_ROOT}  (shared with main pipeline)")
print(f"  Pool size:    {POOL_SIZE}  |  Per-experiment N: {N_EXPERIMENT}")


---
## Models / Environment

Run once per Colab session.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Mount Drive + install packages                     ║
# ╚══════════════════════════════════════════════════════════════╝
from google.colab import drive
import os

drive.mount("/content/drive")

# Only sandbox-specific dirs need creating here.
# The shared pipeline dirs (annotations/, labels/, audio/) already
# exist if 03_Annotation_Pipeline_v2 has been run at least once.
for d in [SANDBOX_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Ensure shared dirs exist too (safe no-op if already present)
for d in [ANNOT_CACHE, AUDIO_CACHE,
          f"{LABELS_ROOT}/boundary/agree",
          f"{LABELS_ROOT}/boundary/disagree",
          f"{LABELS_ROOT}/intonation/agree",
          f"{LABELS_ROOT}/break_idx/agree"]:
    os.makedirs(d, exist_ok=True)

print("✓ Drive mounted. Directories ready.")

!pip install -q praat-parselmouth soundfile evaluate librosa

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ PyTorch {torch.__version__} | Device: {device}")
if not torch.cuda.is_available():
    print("⚠  No GPU detected — PSST will be very slow. Switch to T4 runtime.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║ CELL 3 - Wav2ToBI (assumes it's installed)                   ║
# ╚══════════════════════════════════════════════════════════════╝
import os, re

MODEL_PY = f"{WAV2TOBI_DIR}/src/model.py"
with open(MODEL_PY) as f: src = f.read()
match = re.search(r'self\.lstm_hidden_size = (\d+)', src)
if match and int(match.group(1)) != 256:
    src = re.sub(r'self\.lstm_hidden_size = \d+',
                 'self.lstm_hidden_size = 256', src)
    with open(MODEL_PY, "w") as f: f.write(src)
    print("✓ Patched model.py: lstm_hidden_size → 256")
else:
    print("✓ model.py already patched")

print("\n✓ Wav2ToBI ready.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Load models                                       ║
# ╚══════════════════════════════════════════════════════════════╝
import sys, importlib, torch
from transformers import (WhisperForConditionalGeneration, WhisperProcessor,
                          Wav2Vec2FeatureExtractor)

BOUNDARY_TOKEN = "!!!!!"

print(f"Loading PSST from {PSST_CKPT}...")
psst_processor = WhisperProcessor.from_pretrained(PSST_CKPT)
psst_model     = WhisperForConditionalGeneration.from_pretrained(PSST_CKPT).to(device)
psst_model.eval()
print("✓ PSST loaded.")

W2T_SRC = f"{WAV2TOBI_DIR}/src"
if W2T_SRC not in sys.path:
    sys.path.insert(0, W2T_SRC)
import model as _w2t_module
importlib.reload(_w2t_module)
from model import Wav2Vec2ForAudioFrameClassification_custom

print(f"Loading Wav2ToBI from {WAV2TOBI_CKPT}...")
w2t_processor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-base")
w2t_model     = Wav2Vec2ForAudioFrameClassification_custom.from_pretrained(
    WAV2TOBI_CKPT, ignore_mismatched_sizes=True).to(device)
w2t_model.eval()
print(f"✓ Wav2ToBI loaded. lstm_hidden_size = {w2t_model.lstm_hidden_size}")

import torchaudio
print("Loading CTC alignment model...")
FA_BUNDLE   = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
fa_model    = FA_BUNDLE.get_model().to(device)
fa_labels   = FA_BUNDLE.get_labels()
fa_model.eval()
FA_STRIDE_S = 320 / 16000
print("✓ CTC model loaded.")


---
## Helper Functions

**Key design:** PSST results are read from `annotations/{sample_id}.json` — the
same cache the main pipeline writes. PSST is never re-run in the sandbox.
Only Wav2ToBI peak detection and F0 classification are re-executed per sweep,
and Wav2ToBI's forward pass runs **once per sample** regardless of how many
parameter values are being tested.


### CTC Alignment (audio to text)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — CTC alignment helpers (map audio to text)          ║
# ╚══════════════════════════════════════════════════════════════╝
import re, torch, numpy as np, librosa
from difflib import SequenceMatcher

def text_to_tokens(text, labels):
    label2idx = {l: i for i, l in enumerate(labels)}
    tokens = []
    for word in text.upper().split():
        for char in word:
            if char in label2idx:
                tokens.append(label2idx[char])
        tokens.append(label2idx['|'])
    return tokens

def get_trellis(emission, tokens, blank_id=0):
    T, _ = emission.shape
    N = len(tokens)
    trellis = torch.full((T + 1, N + 1), -float("inf"))
    trellis[0, 0] = 0.0
    for t in range(T):
        trellis[t+1, 1:] = torch.maximum(
            trellis[t, 1:]  + emission[t, blank_id],
            trellis[t, :-1] + emission[t, tokens])
        trellis[t+1, 1:] = torch.maximum(
            trellis[t+1, 1:],
            trellis[t, 1:] + emission[t, tokens])
    return trellis

def backtrack(trellis, emission, tokens, blank_id=0):
    t = trellis.size(0) - 1
    j = trellis.size(1) - 1
    path = [(j - 1, t - 1)]
    while j > 1:
        if (trellis[t-1, j-1] + emission[t-1, tokens[j-2]]
                > trellis[t-1, j] + emission[t-1, blank_id]):
            j -= 1
        t -= 1
        path.append((j - 1, t - 1))
    return path[::-1]

def merge_to_words(path, labels, tokens, stride_s, original_words=None):
    words, current_chars, word_start_frame = [], [], None
    for tok_idx, frame in path:
        label = labels[tokens[tok_idx]]
        if label == '|':
            if current_chars:
                wi   = len(words)
                word = (original_words[wi]
                        if original_words and wi < len(original_words)
                        else ''.join(current_chars))
                words.append({'word': word,
                              'start': round(word_start_frame * stride_s, 4),
                              'end':   round(frame * stride_s, 4)})
                current_chars, word_start_frame = [], None
        else:
            if not current_chars: word_start_frame = frame
            current_chars.append(label)
    if current_chars:
        wi   = len(words)
        word = (original_words[wi]
                if original_words and wi < len(original_words)
                else ''.join(current_chars))
        words.append({'word': word,
                      'start': round(word_start_frame * stride_s, 4),
                      'end':   round(path[-1][1] * stride_s, 4)})
    return words

def get_word_timestamps(audio16k, text):
    """Run CTC forced alignment. Returns word timestamp list or [] on failure."""
    text_clean = re.sub(r"[^a-zA-Z\s']", '', text).strip()
    if not text_clean: return []
    try:
        waveform = torch.tensor(audio16k, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            emission, _ = fa_model(waveform)
        emission = torch.log_softmax(emission[0], dim=-1).cpu()
        tokens   = text_to_tokens(text_clean, fa_labels)
        if not tokens: return []
        trellis  = get_trellis(emission, tokens)
        path     = backtrack(trellis, emission, tokens)
        return merge_to_words(path, fa_labels, tokens, FA_STRIDE_S,
                               original_words=text.split())
    except Exception as e:
        print(f"    ⚠ CTC alignment failed: {e}")
        return []

print("✓ Cell 5: CTC helpers defined.")




### PSST

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — PSST helpers                                      ║
# ╚══════════════════════════════════════════════════════════════╝
import json, os, re, torch
import numpy as np, librosa
from difflib import SequenceMatcher


def run_psst(audio_array, sr=16000):
    """Run PSST on a 16 kHz numpy array. Returns raw string with !!!!! markers."""
    if sr != 16000:
        audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=16000)
    inputs = psst_processor(audio_array, sampling_rate=16000,
                             return_tensors="pt").to(device)
    tokenizer  = psst_processor.tokenizer
    prefix_ids = [
        tokenizer.convert_tokens_to_ids("<|startoftranscript|>"),
        tokenizer.convert_tokens_to_ids("<|en|>"),
        tokenizer.convert_tokens_to_ids("<|transcribe|>"),
        tokenizer.convert_tokens_to_ids("<|notimestamps|>"),
    ]
    with torch.no_grad():
        generated = psst_model.generate(
            inputs.input_features,
            decoder_input_ids=torch.tensor([prefix_ids], device=device),
            suppress_tokens=[],
            max_new_tokens=444,
        )
    return psst_processor.batch_decode(generated, skip_special_tokens=False)[0]


def parse_psst_output(psst_text):
    text = psst_text
    for tok in ["<|startoftranscript|>", "<|transcribe|>", "<|en|>",
                "<|notimestamps|>", "<|endoftext|>"]:
        text = text.replace(tok, "")
    segments = text.strip().split(BOUNDARY_TOKEN)
    return [([w for w in seg.strip().split() if w], i < len(segments) - 1)
            for i, seg in enumerate(segments)]


def align_word_lists(psst_words, mfa_words):
    def norm(w): return re.sub(r"[^a-z']", "", w.lower())
    psst_n = [norm(w) for w in psst_words]
    mfa_n  = [norm(w) for w in mfa_words]
    matcher = SequenceMatcher(None, psst_n, mfa_n, autojunk=False)
    mapping = {}
    for op, i1, i2, j1, j2 in matcher.get_opcodes():
        if op == 'equal':
            for k in range(i2 - i1):
                mapping[i1 + k] = j1 + k
        elif len(range(i1, i2)) > 0 and len(range(j1, j2)) > 0:
            for k, pi in enumerate(range(i1, i2)):
                mi = j1 + round(k / max(i2-i1-1, 1) * max(j2-j1-1, 0))
                mapping[pi] = min(mi, j2 - 1)
    return mapping


def psst_to_word_boundaries(psst_text, mfa_words):
    """Convert PSST raw output to MFA word indices where boundaries occur."""
    segments = parse_psst_output(psst_text)
    all_words, boundary_indices = [], []
    for words, has_boundary in segments:
        all_words.extend(words)
        if has_boundary and words:
            boundary_indices.append(len(all_words) - 1)
    if not all_words: return []
    mapping = align_word_lists(all_words, [w['word'] for w in mfa_words])
    result = []
    for pi in boundary_indices:
        if pi in mapping:
            result.append(mapping[pi])
        else:
            prop = pi / max(len(all_words) - 1, 1)
            result.append(round(prop * (len(mfa_words) - 1)))
    return sorted(set(result))


def load_annotation_cache(sample_id):
    """Load the main pipeline annotation cache for a sample. Returns None if absent."""
    path = os.path.join(ANNOT_CACHE, f"{sample_id}.json")
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None


def save_annotation_cache(sample_id, psst_raw, mfa_words, text):
    """
    Write a minimal annotation cache entry for a sample that hasn't been
    processed by the main pipeline yet.

    Format matches what 03_Annotation_Pipeline_v2 writes, so the main
    pipeline will recognise it as already-cached on its next run.
    """
    data = {
        'sample_id': sample_id,
        'text':      text,
        'psst_raw':  psst_raw,
        'words': [
            {'word': w['word'], 'start_s': w['start'], 'end_s': w['end'],
             'psst_boundary': False, 'wav2tobi_boundary': False,
             'consensus_boundary': False, 'break_idx': None,
             'intonation': 'none', 'agreement_status': 'agree_none'}
            for w in mfa_words
        ],
        'meta': {
            'psst_n_boundaries': 0, 'wav2tobi_n_boundaries': 0,
            'consensus_n_boundaries': 0, 'n_psst_only': 0,
            'n_wav2tobi_only': 0, 'agreement_rate': 0.0,
            'source': 'sandbox'
        }
    }
    with open(os.path.join(ANNOT_CACHE, f"{sample_id}.json"), 'w') as f:
        json.dump(data, f, indent=2)

def load_psst_cache(sample_id):
    """
    Load cached PSST result for a sample from psst_cache/.
    Returns dict with keys: sample_id, text, psst_raw, mfa_words.
    Returns None if no cache file exists yet.
    """
    path = f"{PSST_CACHE}/{sample_id}.json"
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None


def save_psst_cache(sample_id, psst_raw, mfa_words, text):
    """
    Save PSST raw output and word timestamps for a sample to psst_cache/.
    These are the only two things that never need to be recomputed —
    PSST has no tunable parameters, and mfa_words come from CTC alignment
    which is also parameter-free.
    """
    os.makedirs(PSST_CACHE, exist_ok=True)
    path = f"{PSST_CACHE}/{sample_id}.json"
    with open(path, 'w') as f:
        json.dump({
            'sample_id': sample_id,
            'text':      text,
            'psst_raw':  psst_raw,
            'mfa_words': mfa_words,
        }, f, indent=2)

print("✓ Cell 6: PSST helpers defined.")

### Wav2ToBI / F0

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Wav2ToBI + F0 helpers (parameterised)             ║
# ║                                                              ║
# ║  Every function that has a tunable hyperparameter takes it  ║
# ║  as an explicit argument with the default as fallback.      ║
# ║  This makes sweep functions clean — just pass the value.    ║
# ╚══════════════════════════════════════════════════════════════╝
import torch, parselmouth
import numpy as np, librosa
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d


def run_wav2tobi_raw(audio_array, sr=16000):
    """
    Run the Wav2ToBI forward pass and return raw logits + F0 stats.

    Returns dict with keys: logits, mean, std, frame_s
    Separating this from peak detection means we can run inference once
    and then sweep threshold multipliers cheaply without re-running the model.
    """
    if sr != 16000:
        audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=16000)
        sr = 16000

    snd = parselmouth.Sound(audio_array, sr)
    pitch_values = snd.to_pitch(time_step=0.02, pitch_floor=75,
                                 pitch_ceiling=500).selected_array['frequency']
    pitch_tensor = torch.tensor(pitch_values, dtype=torch.float32)

    inputs = w2t_processor(audio_array, sampling_rate=sr,
                            return_tensors="pt", padding=True)
    with torch.no_grad():
        outputs = w2t_model(inputs.input_values.to(device), pitch=[pitch_tensor])

    logits = outputs.logits.squeeze().cpu().numpy()
    if logits.ndim > 1: logits = logits[:, 0]

    return {
        'logits':  logits,
        'mean':    float(logits.mean()),
        'std':     float(logits.std()),
        'frame_s': 320 / sr,
    }


def detect_peaks_from_logits(raw, threshold_mult=DEFAULT_W2T_THRESHOLD_MULT,
                              high_mult=DEFAULT_W2T_HIGH_MULT):
    """
    Detect phrase boundary peaks from pre-computed Wav2ToBI logits.

    threshold_mult: peak must exceed mean + threshold_mult * std
    high_mult:      peaks above mean + high_mult * std → break index 4,
                    others → break index 3

    Returns list of [timestamp_s, break_index_str] pairs.
    """
    logits   = raw['logits']
    mean, std, frame_s = raw['mean'], raw['std'], raw['frame_s']
    smoothed  = gaussian_filter1d(logits.astype(float), sigma=3)
    threshold = max(mean + threshold_mult * std, 0.1)
    high_thresh = mean + high_mult * std

    peaks, _ = find_peaks(smoothed, height=threshold,
                           distance=int(0.2 / frame_s))
    return [[round(idx * frame_s, 3),
             "4" if smoothed[idx] >= high_thresh else "3"]
            for idx in peaks]


def wav2tobi_to_word_boundaries(detections, mfa_words,
                                 tolerance_s=DEFAULT_ALIGNMENT_TOLERANCE_S):
    """Map Wav2ToBI detections to MFA word indices within tolerance_s."""
    results = []
    for timestamp, break_idx in detections:
        best_idx = min(range(len(mfa_words)),
                       key=lambda i: abs(mfa_words[i]['end'] - timestamp))
        if abs(mfa_words[best_idx]['end'] - timestamp) <= tolerance_s:
            results.append((best_idx, break_idx))
    return results


def compute_consensus(psst_indices, w2t_pairs,
                       window=DEFAULT_CONSENSUS_WINDOW):
    """Return word indices where PSST and Wav2ToBI agree within window words."""
    psst_set = set(psst_indices)
    w2t_set  = set(idx for idx, _ in w2t_pairs)
    return sorted(pi for pi in psst_set
                  if any(abs(pi - wi) <= window for wi in w2t_set))


def classify_agreement_counts(psst_indices, w2t_indices, n_words,
                               window=DEFAULT_CONSENSUS_WINDOW):
    """Return counts of consensus / psst_only / wav2tobi_only."""
    psst_set = set(psst_indices)
    w2t_exp  = set(wi + d for wi in w2t_indices
                   for d in range(-window, window + 1))
    consensus  = sum(1 for pi in psst_set if pi in w2t_exp)
    psst_only  = sum(1 for pi in psst_set if pi not in w2t_exp)
    w2t_only   = sum(1 for wi in w2t_indices
                     if not any(abs(wi - pi) <= window for pi in psst_set))
    return {
        'total_consensus':     consensus,   # ← was 'agree', renamed for consistency
        'total_psst_only':     psst_only,
        'total_wav2tobi_only': w2t_only,
    }


def classify_intonation(audio_array, sr, word_end_s,
                         window_s=DEFAULT_F0_WINDOW_S,
                         slope_thresh=DEFAULT_F0_SLOPE_HZ):
    """Classify F0 direction at a boundary. Returns rising/falling/level/unclear."""
    start_s = max(0.0, word_end_s - window_s)
    segment = audio_array[int(start_s * sr): int(word_end_s * sr)]
    if len(segment) < int(0.05 * sr): return 'unclear'
    try:
        f0 = parselmouth.Sound(segment, sr).to_pitch(
            time_step=0.01, pitch_floor=75, pitch_ceiling=500
        ).selected_array['frequency']
        f0v = f0[f0 > 0]
    except Exception:
        return 'unclear'
    if len(f0v) < 4: return 'unclear'
    mid = len(f0v) // 2
    slope = np.mean(f0v[mid:]) - np.mean(f0v[:mid])
    if slope >  slope_thresh: return 'rising'
    if slope < -slope_thresh: return 'falling'
    return 'level'


print("✓ Cell 7: Wav2ToBI + F0 helpers defined.")

### Samples

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Pool management + data fetching                   ║
# ╚══════════════════════════════════════════════════════════════╝
import json, os, random, numpy as np, librosa, soundfile as sf
from datasets import load_dataset

POOL_FILE = f"{SANDBOX_DIR}/sandbox_ids.json"


def build_or_load_pool():
    """
    Load the pool from sandbox_ids.json, or build a new one by streaming
    POOL_SIZE IDs from LibriTTS. Only IDs are stored — no audio, no annotations.
    """
    if os.path.exists(POOL_FILE):
        with open(POOL_FILE) as f:
            pool = json.load(f)
        print(f"✓ Loaded existing pool: {len(pool)} IDs from {POOL_FILE}")
        return pool

    print(f"Building new pool of {POOL_SIZE} IDs from LibriTTS...")
    random.seed(RANDOM_SEED)
    split_name = LIBRITTS_SPLIT.replace('-', '.')
    ds = load_dataset("blabble-io/libritts", "clean", split=split_name,
                      streaming=True, trust_remote_code=False)
    reservoir = []
    for i, sample in enumerate(ds):
        if i < POOL_SIZE:
            reservoir.append(sample['id'])
        else:
            j = random.randint(0, i)
            if j < POOL_SIZE:
                reservoir[j] = sample['id']
        if i >= 5000 - 1:
            break
    with open(POOL_FILE, 'w') as f:
        json.dump(reservoir, f, indent=2)
    print(f"✓ Pool built and saved: {len(reservoir)} IDs → {POOL_FILE}")
    return reservoir


def get_experiment_ids():
    """
    Return IDs for this experiment run, respecting USE_MANUAL_IDS.

    - False  → first N_EXPERIMENT IDs from pool
    - True + MANUAL_IDS_FROM  → IDs from a previous results file
    - True + MANUAL_IDS       → literal list from config
    """
    if USE_MANUAL_IDS:
        if MANUAL_IDS_FROM:
            with open(MANUAL_IDS_FROM) as f:
                prev = json.load(f)
            ids = prev['ids_used']
            print(f"✓ Loaded {len(ids)} IDs from previous run: {MANUAL_IDS_FROM}")
            return ids
        if MANUAL_IDS:
            print(f"✓ Using {len(MANUAL_IDS)} manually specified IDs.")
            return MANUAL_IDS
        raise ValueError(
            "USE_MANUAL_IDS=True but MANUAL_IDS and MANUAL_IDS_FROM are both empty.")
    pool = build_or_load_pool()
    ids  = pool[:N_EXPERIMENT]
    print(f"✓ Using first {len(ids)} IDs from pool.")
    return ids


def fetch_sample_audio(sample_id):
    """
    Stream audio for one sample ID from LibriTTS.
    Returns (audio_array, sr, text) or None on failure.
    """
    split_name = LIBRITTS_SPLIT.replace('-', '.')
    ds = load_dataset("blabble-io/libritts", "clean", split=split_name,
                      streaming=True, trust_remote_code=False)
    for sample in ds:
        if sample['id'] == sample_id:
            return (np.array(sample['audio']['array']),
                    sample['audio']['sampling_rate'],
                    sample['text_normalized'])
    return None


def get_or_build_sample(sample_id):
    """
    Return (psst_raw, mfa_words, audio16, text) for a sample.

    Reads PSST data from the shared annotations/ cache if present.
    If not, fetches audio, runs CTC alignment and PSST, and writes
    the result to annotations/ so the main pipeline can reuse it.

    Always fetches audio fresh (for Wav2ToBI + F0 — cannot be cached
    without keeping the file, which KEEP_AUDIO controls).

    Returns None if the sample cannot be fetched.
    """
    # ── Try loading PSST data from main pipeline cache ────────────
    cached = load_annotation_cache(sample_id)
    if cached:
        psst_raw  = cached['psst_raw']
        mfa_words = [{'word': w['word'], 'start': w['start_s'], 'end': w['end_s']}
                     for w in cached['words']]
        text      = cached['text']
        have_psst = True
    else:
        have_psst = False

    # ── Always fetch audio (needed for Wav2ToBI + F0) ─────────────
    print(f"  [{sample_id}] Fetching audio...")
    result = fetch_sample_audio(sample_id)
    if result is None:
        print(f"  [{sample_id}] ⚠ Could not fetch audio — skipping.")
        return None
    audio, sr, text = result
    audio16 = librosa.resample(audio, orig_sr=sr, target_sr=16000) \
              if sr != 16000 else audio.copy()

    if KEEP_AUDIO:
        wav_path = os.path.join(AUDIO_CACHE, f"{sample_id}.wav")
        if not os.path.exists(wav_path):
            sf.write(wav_path, audio, sr)

    # ── Run CTC + PSST if not cached ──────────────────────────────
    if not have_psst:
        mfa_words = get_word_timestamps(audio16, text)
        if not mfa_words:
            print(f"  [{sample_id}] ⚠ CTC alignment failed — skipping.")
            return None
        psst_raw = run_psst(audio16)
        save_annotation_cache(sample_id, psst_raw, mfa_words, text)
        print(f"  [{sample_id}] ✓ Annotated and cached ({len(mfa_words)} words).")
    else:
        print(f"  [{sample_id}] ✓ Loaded from cache ({len(mfa_words)} words).")

    return psst_raw, mfa_words, audio16, text

def get_or_build_psst_cache(sample_id):
    """
    Return PSST cache for a sample, running inference if not yet cached.

    If the cache exists on Drive, loads and returns it immediately.
    Otherwise: fetches audio, runs CTC alignment, runs PSST, saves cache.

    Returns dict with keys: psst_raw, mfa_words, text.
    Returns None if audio cannot be fetched or CTC alignment fails.
    """
    cached = load_psst_cache(sample_id)
    if cached:
        return cached

    print(f"  [{sample_id}] No PSST cache — fetching audio and running inference...")
    result = fetch_sample_audio(sample_id)
    if result is None:
        print(f"  [{sample_id}] ⚠ Could not fetch audio — skipping.")
        return None
    audio, sr, text = result

    audio16 = librosa.resample(audio, orig_sr=sr, target_sr=16000) \
              if sr != 16000 else audio.astype(np.float32)

    mfa_words = get_word_timestamps(audio16, text)
    if not mfa_words:
        print(f"  [{sample_id}] ⚠ CTC alignment failed — skipping.")
        return None

    psst_raw = run_psst(audio16)
    save_psst_cache(sample_id, psst_raw, mfa_words, text)
    print(f"  [{sample_id}] ✓ PSST cached ({len(mfa_words)} words).")
    return load_psst_cache(sample_id)

def load_audio_for_sample(sample_id):
    """
    Load audio for a sample, using Drive cache if available.
    Checks sandbox/audio/{sample_id}.wav first to avoid re-streaming.
    Falls back to fetch_sample_audio() and caches the result.
    Returns (audio_float32, sr, text) or None on failure.
    """
    import soundfile as sf

    audio_dir  = f"{SANDBOX_ROOT}/audio"
    audio_path = f"{audio_dir}/{sample_id}.wav"
    os.makedirs(audio_dir, exist_ok=True)

    if os.path.exists(audio_path):
        audio, sr = sf.read(audio_path)
        # text is not stored in the wav — retrieve from PSST cache if available
        cached = load_psst_cache(sample_id)
        text = cached['text'] if cached else ""
        return audio.astype(np.float32), sr, text

    # Not cached — stream and save
    result = fetch_sample_audio(sample_id)
    if result is None:
        return None
    audio, sr, text = result

    sf.write(audio_path, audio, sr)
    return audio.astype(np.float32), sr, text

print("✓ Cell 8: Pool management + data helpers defined.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Result recording + sweep engine                   ║
# ╚══════════════════════════════════════════════════════════════╝
import json, numpy as np
from datetime import datetime


def save_experiment_results(experiment_name, param_name,
                             results_by_value, ids_used):
    """Write experiment results to sandbox/results/."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename  = f"{RESULTS_DIR}/{experiment_name}_{timestamp}.json"
    with open(filename, 'w') as f:
        json.dump({
            'experiment': experiment_name,
            'param_name': param_name,
            'timestamp':  timestamp,
            'n_samples':  len(ids_used),
            'ids_used':   ids_used,
            'results':    {str(v): s for v, s in results_by_value.items()},
        }, f, indent=2)
    print(f"\n✓ Results saved → {filename}")
    return filename


def print_sweep_table(param_name, results_by_value):
    """Print a formatted summary table for a parameter sweep."""
    print(f"\n{'='*72}")
    print(f"  SWEEP: {param_name}")
    print(f"{'='*72}")
    print(f"  {'Value':>8}  {'Consensus':>9}  {'PSST-only':>9}  "
          f"{'W2T-only':>8}  {'Agree%':>7}  {'Rising%':>7}  {'Fall%':>6}  {'Level%':>7}")
    print(f"  {'-'*8}  {'-'*9}  {'-'*9}  {'-'*8}  {'-'*7}  {'-'*7}  {'-'*6}  {'-'*7}")
    for v, s in sorted(results_by_value.items(), key=lambda x: float(str(x[0]))):
        total_psst  = s['total_consensus'] + s['total_psst_only']
        agree_pct   = 100 * s['total_consensus']  / max(total_psst, 1)
        rising_pct  = 100 * s['inton_rising']     / max(s['total_consensus'], 1)
        falling_pct = 100 * s['inton_falling']    / max(s['total_consensus'], 1)
        level_pct   = 100 * s['inton_level']      / max(s['total_consensus'], 1)
        default_marker = " ←" if v == _get_default(param_name) else ""
        print(f"  {str(v):>8}  {s['total_consensus']:>9}  {s['total_psst_only']:>9}  "
              f"{s['total_wav2tobi_only']:>8}  {agree_pct:>6.1f}%  "
              f"{rising_pct:>6.1f}%  {falling_pct:>5.1f}%  "
              f"{level_pct:>6.1f}%{default_marker}")
    print(f"{'='*72}")
    print("  ← = current default")


def _get_default(param_name):
    """Return the current default value for a parameter, for table annotation."""
    return {
        'ALIGNMENT_TOLERANCE_S': DEFAULT_ALIGNMENT_TOLERANCE_S,
        'W2T_THRESHOLD_MULT':    DEFAULT_W2T_THRESHOLD_MULT,
        'F0_WINDOW_S':           DEFAULT_F0_WINDOW_S,
        'F0_SLOPE_HZ':           DEFAULT_F0_SLOPE_HZ,
    }.get(param_name)

def _finalise_stats(s):
    """Add mean/std agreement rate to a stats dict in-place."""
    rates = s.get('agreement_rates', [])
    s['mean_agreement_rate'] = float(np.mean(rates)) if rates else 0.0
    s['std_agreement_rate']  = float(np.std(rates))  if rates else 0.0
    return s

def run_sweep(experiment_name, param_name, param_values, apply_fn, ids=None):
    """
    Run a hyperparameter sweep over param_values on a fixed set of samples.

    apply_fn(raw_logits, psst_raw, audio16, mfa_words, psst_indices, param_value)
    → dict with keys: agree, psst_only, wav2tobi_only,
                      rising, falling, level, unclear

    Wav2ToBI's forward pass runs once per sample regardless of how many
    param_values are swept — only peak detection is re-run per value.
    """
    if ids is None:
        ids = get_experiment_ids()

    results_by_value = {v: {
        'total_consensus': 0, 'total_psst_only': 0,
        'total_wav2tobi_only': 0, 'total_words': 0,
        'inton_rising': 0, 'inton_falling': 0,
        'inton_level': 0, 'inton_unclear': 0,
        'agreement_rates': [],
    } for v in param_values}

    ids_used, n_failed = [], 0

    print(f"\nRunning sweep: {experiment_name}")
    print(f"  Parameter: {param_name}  Values: {param_values}")
    print(f"  Samples:   {len(ids)}")
    print()

    for i, sample_id in enumerate(ids):
        print(f"[{i+1}/{len(ids)}] {sample_id}")

        data = get_or_build_sample(sample_id)
        if data is None:
            n_failed += 1
            continue

        psst_raw, mfa_words, audio16, text = data
        psst_indices = psst_to_word_boundaries(psst_raw, mfa_words)

        # Run Wav2ToBI inference once, sweep params against the same logits
        raw = run_wav2tobi_raw(audio16)

        for v in param_values:
            s = apply_fn(raw, psst_raw, audio16, mfa_words, psst_indices, v)
            agg = results_by_value[v]
            agg['total_consensus']     += s['total_consensus']
            agg['total_psst_only']     += s['total_psst_only']
            agg['total_wav2tobi_only'] += s['total_wav2tobi_only']
            agg['total_words']         += len(mfa_words)
            agg['inton_rising']        += s.get('rising',  0)
            agg['inton_falling']       += s.get('falling', 0)
            agg['inton_level']         += s.get('level',   0)
            agg['inton_unclear']       += s.get('unclear', 0)
            n_psst = s['total_consensus'] + s['total_psst_only']
            agg['agreement_rates'].append(s['total_consensus'] / max(n_psst, 1))

        ids_used.append(sample_id)
        print(f"  ✓")

    # Compute mean/std agreement rate per value
    for v in param_values:
        _finalise_stats(results_by_value[v])

    print(f"\nSweep complete. {len(ids_used)} samples, {n_failed} failed.")
    save_experiment_results(experiment_name, param_name, results_by_value, ids_used)
    try:
        print_sweep_table(param_name, results_by_value)
    except Exception as e:
        print(f"  ⚠ Table print failed ({e}) — results already saved, safe to continue.")
    return results_by_value, ids_used


print("✓ Cell 9: Sweep engine defined.")


---
## Pre-Experiment Setups

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Build / load the sample pool                     ║
# ╚══════════════════════════════════════════════════════════════╝
pool = build_or_load_pool()
print(f"\nPool preview (first 5): {pool[:5]}")


---
## Experiments

Each experiment is self-contained — run them independently in any order.
All write a timestamped JSON to `sandbox/results/`.

The default parameter value is marked with ← in each output table.


### Alignment Tolerance and Threshold

**Parameter:** `ALIGNMENT_TOLERANCE_S`  
How close (in seconds) must a Wav2ToBI peak be to a word's end time
to be assigned to that word?

Too tight → genuine matches discarded (agreement drops, wav2tobi_only rises).  
Too loose → wrong words get boundaries (spurious agreement).  

**Parameter:** `W2T_THRESHOLD_MULT` (the `k` in `mean + k × std`)  
How sensitive should Wav2ToBI's peak detector be?

Lower k → more peaks, higher recall, more wav2tobi_only noise.  
Higher k → fewer peaks, higher precision, may miss real boundaries.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  DIAGNOSTIC — Wav2ToBI peak offset relative to word ends    ║
# ║                                                              ║
# ║  Run this BEFORE the sweeps to understand where Wav2ToBI    ║
# ║  peaks tend to fall relative to word-end timestamps.        ║
# ║  The distribution informs what tolerance range makes sense. ║
# ╚══════════════════════════════════════════════════════════════╝
import matplotlib.pyplot as plt

DIAG_GENEROUS_WINDOW = 0.5   # cast wide net — this is diagnostic only
DIAG_N = N_EXPERIMENT        # use the same N as experiments

def diagnose_peak_offsets(sample_ids=None):
    """
    For every Wav2ToBI peak that can be matched to a word within a generous
    window, record (peak_time - word_end_time).

    Negative offset  → peak fires BEFORE the word ends (common for prosodic cues
                        like lengthening that begin inside the final segment).
    Positive offset  → peak fires AFTER the word ends (pause detection, etc.).

    The resulting distribution tells you:
      - Whether to use a symmetric or asymmetric tolerance window
      - What the upper bound on a justifiable tolerance value is
    """
    if sample_ids is None:
        sample_ids = pool[:DIAG_N]

    offsets     = []
    n_processed = 0
    n_failed    = 0

    print(f"Running peak-offset diagnostic on {len(sample_ids)} samples...\n")

    for i, sample_id in enumerate(sample_ids):
        print(f"  [{i+1}/{len(sample_ids)}] {sample_id}", end="  ")

        result = load_audio_for_sample(sample_id)
        if result is None:
            print("⚠ audio unavailable, skipping")
            n_failed += 1
            continue
        audio, sr, _ = result
        audio16 = librosa.resample(audio, orig_sr=sr, target_sr=16000) \
                  if sr != 16000 else audio

        # PSST cache gives us mfa_words (word-end timestamps)
        psst_cache = get_or_build_psst_cache(sample_id)
        if psst_cache is None:
            print("⚠ PSST cache failed, skipping")
            n_failed += 1
            continue
        mfa_words = psst_cache['mfa_words']

        # Raw Wav2ToBI logits → peak timestamps (no threshold filtering yet —
        # we use detect_peaks_from_logits with a very low threshold to see all peaks)
        raw = run_wav2tobi_raw(audio16)
        detections = detect_peaks_from_logits(raw, threshold_mult=0.5)  # permissive

        sample_offsets = []
        for peak_t in detections:
            # Nearest word end within generous window
            peak_t = peak_t[0] # tuple: (timestamp + break index, unpairing)
            best_word = None
            best_dist = float('inf')
            for w in mfa_words:
                dist = abs(peak_t - w['end'])
                if dist < DIAG_GENEROUS_WINDOW and dist < best_dist:
                    best_dist = dist
                    best_word = w
            if best_word is not None:
                sample_offsets.append(peak_t - best_word['end'])

        offsets.extend(sample_offsets)
        n_processed += 1
        print(f"{len(detections)} peaks → {len(sample_offsets)} matched")

    if not offsets:
        print("\n⚠ No offsets collected — check that models are loaded and pool is built.")
        return None

    offsets = np.array(offsets)

    # ── Statistics ──────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  PEAK OFFSET STATISTICS  (peak_time − word_end_time)")
    print(f"{'='*60}")
    print(f"  Samples processed : {n_processed}  |  failed: {n_failed}")
    print(f"  Total peaks matched: {len(offsets)}")
    print()
    print(f"  mean   : {offsets.mean()*1000:+7.1f} ms")
    print(f"  median : {np.median(offsets)*1000:+7.1f} ms")
    print(f"  std    : {offsets.std()*1000:7.1f} ms")
    print(f"  p5     : {np.percentile(offsets,5)*1000:+7.1f} ms")
    print(f"  p95    : {np.percentile(offsets,95)*1000:+7.1f} ms")
    print(f"  min    : {offsets.min()*1000:+7.1f} ms")
    print(f"  max    : {offsets.max()*1000:+7.1f} ms")
    print()
    print(f"  Before word end (offset < 0) : {100*(offsets < 0).mean():.1f}%")
    print(f"  At word end     (|offset|<20ms): {100*(np.abs(offsets)<0.02).mean():.1f}%")
    print(f"  After  word end (offset > 0) : {100*(offsets > 0).mean():.1f}%")
    print()

    # Suggested tolerance: covers 90% of matches (p5–p95 range, floored at 0)
    suggested = max(abs(np.percentile(offsets, 5)),
                    abs(np.percentile(offsets, 95)))
    print(f"  ➜  Suggested tolerance (covers ~90% of matches): {suggested*1000:.0f} ms")
    print(f"     (= abs(max(p5, p95)) — use as upper bound in your sweep)")
    print(f"{'='*60}")

    # ── Histogram ───────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.hist(offsets * 1000, bins=50, edgecolor='black', alpha=0.8)
    ax.axvline(0,                        color='red',    linestyle='--', lw=1.5,
               label='word end (t=0)')
    ax.axvline(offsets.mean()*1000,      color='orange', linestyle='--', lw=1.5,
               label=f'mean ({offsets.mean()*1000:+.0f} ms)')
    ax.axvline(np.percentile(offsets,5)*1000,  color='steelblue', linestyle=':',
               lw=1.2, label=f'p5 ({np.percentile(offsets,5)*1000:+.0f} ms)')
    ax.axvline(np.percentile(offsets,95)*1000, color='steelblue', linestyle=':',
               lw=1.2, label=f'p95 ({np.percentile(offsets,95)*1000:+.0f} ms)')
    ax.set_xlabel("Offset (ms)   [negative = peak before word end]")
    ax.set_ylabel("Count")
    ax.set_title("Wav2ToBI peak offset relative to word-end timestamps")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    return offsets

diag_offsets = diagnose_peak_offsets()

In [ ]:
diag_offsets
1-sum(1 for x in diag_offsets if x < -0.15 or x > 0.15)/len(diag_offsets)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  JOINT SWEEP — ALIGNMENT_TOLERANCE_S × W2T_THRESHOLD_MULT  ║
# ║                                                              ║
# ║  These two parameters interact: a low threshold produces    ║
# ║  more peaks, and a loose tolerance is more likely to match  ║
# ║  them to PSST boundaries. Sweeping them independently can   ║
# ║  hide this interaction. This cell sweeps them jointly.      ║
# ║                                                              ║
# ║  Results are saved as a single JSON with nested structure:  ║
# ║    results[tolerance][threshold_mult] = stats_dict          ║
# ╚══════════════════════════════════════════════════════════════╝
import itertools
from datetime import datetime

JOINT_TOLERANCES  = [0.15] # <- covers 96% via diagnostic
JOINT_THRESHOLDS  = [0.75, 1.0,  1.25, 1.5,  2.0,  2.5 ]


def _empty_joint_stats():
    return {
        'total_consensus': 0, 'total_psst_only': 0,
        'total_wav2tobi_only': 0, 'total_words': 0,
        'inton_rising': 0, 'inton_falling': 0,
        'inton_level': 0, 'inton_unclear': 0,
        'agreement_rates': [],
    }


def save_joint_results(tolerances, thresholds, results_2d, ids_used):
    """
    Save 2D sweep results to a single JSON.

    results_2d[tol][mult] = stats_dict
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename  = f"{RESULTS_DIR}/joint_tol_x_thresh_{timestamp}.json"
    payload   = {
        'experiment':   'joint_tolerance_threshold',
        'timestamp':    timestamp,
        'n_samples':    len(ids_used),
        'ids_used':     ids_used,
        'param_axes':   ['ALIGNMENT_TOLERANCE_S', 'W2T_THRESHOLD_MULT'],
        'axis_values':  {'tolerances': tolerances, 'thresholds': thresholds},
        'results': {
            str(tol): {
                str(mult): {k: v for k, v in results_2d[tol][mult].items()
                            if k != 'agreement_rates'}   # skip raw list to keep JSON slim
                for mult in thresholds
            }
            for tol in tolerances
        },
    }
    with open(filename, 'w') as f:
        json.dump(payload, f, indent=2)
    print(f"\n✓ Joint results saved → {filename}")
    return filename


def print_joint_table(tolerances, thresholds, results_2d):
    """Print agree% as a 2D grid (tolerances = rows, thresholds = cols)."""
    col_w = 10
    header = f"  {'tol \\ mult':>12}" + "".join(f"{str(m):>{col_w}}" for m in thresholds)
    print(f"\n{'='*len(header)}")
    print("  JOINT SWEEP — agree% (consensus / (consensus + psst_only))")
    print(f"{'='*len(header)}")
    print(header)
    print(f"  {'-'*12}" + "-"*col_w*len(thresholds))
    for tol in tolerances:
        row = f"  {str(tol):>12}"
        for mult in thresholds:
            s = results_2d[tol][mult]
            total_psst = s['total_consensus'] + s['total_psst_only']
            pct = 100 * s['total_consensus'] / max(total_psst, 1)
            row += f"{pct:>{col_w-1}.1f}%"
        print(row)
    print(f"{'='*len(header)}")


def run_joint_sweep(tolerances=JOINT_TOLERANCES, thresholds=JOINT_THRESHOLDS,
                     ids=None):
    """
    Sweep tolerance × threshold_mult jointly.

    For efficiency:
      - Audio is loaded once per sample (Drive-cached).
      - PSST is loaded/built once per sample.
      - Wav2ToBI forward pass runs once per sample.
      - All (tol, mult) combinations are applied to the same logits.

    Returns (results_2d, ids_used, best_tol, best_mult).
    """
    if ids is None:
        ids = pool[:N_EXPERIMENT]

    # Initialise 2D results dict
    results_2d = {
        tol: {mult: _empty_joint_stats() for mult in thresholds}
        for tol in tolerances
    }
    ids_used = []
    n_failed = 0

    combos = list(itertools.product(tolerances, thresholds))
    print(f"\nJoint sweep: {len(tolerances)} tolerances × {len(thresholds)} thresholds"
          f" = {len(combos)} combinations")
    print(f"Samples: {len(ids)}\n")

    for i, sample_id in enumerate(ids):
        print(f"[{i+1}/{len(ids)}] {sample_id}")

        # ── PSST cache ───────────────────────────────────────────────
        psst_cache = get_or_build_psst_cache(sample_id)
        if psst_cache is None:
            n_failed += 1
            continue
        mfa_words    = psst_cache['mfa_words']
        psst_indices = psst_to_word_boundaries(psst_cache['psst_raw'], mfa_words)

        # ── Audio ────────────────────────────────────────────────────
        result = load_audio_for_sample(sample_id)
        if result is None:
            print(f"  ⚠ Audio unavailable — skipping")
            n_failed += 1
            continue
        audio, sr, _ = result
        audio16 = librosa.resample(audio, orig_sr=sr, target_sr=16000) \
                  if sr != 16000 else audio

        # ── Wav2ToBI forward pass (once) ─────────────────────────────
        raw = run_wav2tobi_raw(audio16)

        # ── Apply all combinations to same logits ────────────────────
        for tol, mult in combos:
            detections  = detect_peaks_from_logits(raw, threshold_mult=mult)
            w2t_pairs   = wav2tobi_to_word_boundaries(detections, mfa_words,
                                                       tolerance_s=tol)
            w2t_indices = [idx for idx, _ in w2t_pairs]
            consensus   = compute_consensus(psst_indices, w2t_pairs)
            counts      = classify_agreement_counts(psst_indices, w2t_indices,
                                                     len(mfa_words))

            inton = {'rising': 0, 'falling': 0, 'level': 0, 'unclear': 0}
            for idx in consensus:
                label = classify_intonation(audio16, 16000, mfa_words[idx]['end'])
                inton[label] += 1

            agg = results_2d[tol][mult]
            agg['total_consensus']     += counts['total_consensus']
            agg['total_psst_only']     += counts['total_psst_only']
            agg['total_wav2tobi_only'] += counts['total_wav2tobi_only']
            agg['total_words']         += len(mfa_words)
            agg['inton_rising']        += inton['rising']
            agg['inton_falling']       += inton['falling']
            agg['inton_level']         += inton['level']
            agg['inton_unclear']       += inton['unclear']
            n_psst = counts['total_consensus'] + counts['total_psst_only']
            agg['agreement_rates'].append(counts['total_consensus'] / max(n_psst, 1))

        ids_used.append(sample_id)
        print(f"  ✓ {len(combos)} combinations applied.")

    # ── Finalise + print ─────────────────────────────────────────────
    for tol in tolerances:
        for mult in thresholds:
            _finalise_stats(results_2d[tol][mult])

    print(f"\n{'='*50}")
    print(f"  Joint sweep complete. {len(ids_used)} samples, {n_failed} failed.")

    print_joint_table(tolerances, thresholds, results_2d)

    # ── Find best (tol, mult) by mean agreement rate ─────────────────
    best_tol, best_mult, best_rate = None, None, -1.0
    for tol in tolerances:
        for mult in thresholds:
            rate = results_2d[tol][mult]['mean_agreement_rate']
            if rate > best_rate:
                best_rate = rate
                best_tol, best_mult = tol, mult

    print(f"\n  ➜  Best combination: tolerance={best_tol}  threshold_mult={best_mult}"
          f"  (mean agree={best_rate:.3f})")
    print(f"     NOTE: pick the plateau, not necessarily the absolute max —")
    print(f"     see the table above for context.")

    save_joint_results(tolerances, thresholds, results_2d, ids_used)
    return results_2d, ids_used, best_tol, best_mult


# Run it
results_joint, ids_joint, best_tol, best_mult = run_joint_sweep()

^ using 1.0 instead because it is a lower threshold with no loss in agreement (more easily justifiable)

3 — F0 Window Size

**Parameter:** `F0_WINDOW_S`  
How many seconds before the boundary word's end to measure F0 slope?

Too short → may miss the contour entirely.  
Too long → picks up F0 from earlier in the phrase, not the boundary.

**Tip:** Set `USE_MANUAL_IDS = True` and listen to the utterances to
verify labels by ear. Use `MANUAL_IDS_FROM` to reuse the same IDs
across Experiments 3 and 4.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  EXPERIMENT 3 — F0 Window Size                              ║
# ╚══════════════════════════════════════════════════════════════╝

F0_WINDOW_VALUES = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]

def apply_f0_window(raw, psst_raw, audio16, mfa_words, psst_indices, window_s):
    detections  = detect_peaks_from_logits(raw)
    w2t_pairs   = wav2tobi_to_word_boundaries(detections, mfa_words)
    w2t_indices = [idx for idx, _ in w2t_pairs]
    counts      = classify_agreement_counts(psst_indices, w2t_indices, len(mfa_words))
    consensus   = compute_consensus(psst_indices, w2t_pairs)
    inton = {'rising': 0, 'falling': 0, 'level': 0, 'unclear': 0}
    for idx in consensus:
        # modified to slope of 2 to match Hz
        inton[classify_intonation(audio16, 16000, mfa_words[idx]['end'],
                                   window_s=window_s, slope_thresh=2.0)] += 1
    return {**counts, **inton}

results_f0_window, ids_f0_window = run_sweep(
    experiment_name = "f0_window",
    param_name      = "F0_WINDOW_S",
    param_values    = F0_WINDOW_VALUES,
    apply_fn        = apply_f0_window,
    ids             = get_experiment_ids() if USE_MANUAL_IDS else None,
)


Tallying the human-annotated results from later in this notebook: 59% falling, 25% rising, 16% flat

Based on this, 0.25 is basically perfect for falling and rising. 0.1 undercounts falling, 0.25 is too large to capture the pattern

In [ ]:
!pip install -q numba --upgrade

### F0 Slope Threshold

**Parameter:** `F0_SLOPE_HZ`  
How large must the F0 slope (Hz) be to classify a boundary as rising or falling?

Too low → everything rising/falling, noise-sensitive.  
Too high → too many 'level' labels (your current run shows 43%).  
Watch how rising% + falling% vs. level% shifts across values.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  EXPERIMENT 4 — F0 Slope Threshold                          ║
# ╚══════════════════════════════════════════════════════════════╝

F0_SLOPE_VALUES = [2.0, 5.0, 7.5, 10.0, 12.5, 15.0, 20.0, 25.0]

def apply_f0_slope(raw, psst_raw, audio16, mfa_words, psst_indices, slope_thresh):
    detections  = detect_peaks_from_logits(raw)
    w2t_pairs   = wav2tobi_to_word_boundaries(detections, mfa_words)
    w2t_indices = [idx for idx, _ in w2t_pairs]
    counts      = classify_agreement_counts(psst_indices, w2t_indices, len(mfa_words))
    consensus   = compute_consensus(psst_indices, w2t_pairs)
    inton = {'rising': 0, 'falling': 0, 'level': 0, 'unclear': 0}
    for idx in consensus:
        inton[classify_intonation(audio16, 16000, mfa_words[idx]['end'],
                                   slope_thresh=slope_thresh)] += 1
    return {**counts, **inton}

results_f0_slope, ids_f0_slope = run_sweep(
    experiment_name = "f0_slope_threshold",
    param_name      = "F0_SLOPE_HZ",
    param_values    = F0_SLOPE_VALUES,
    apply_fn        = apply_f0_slope,
    ids             = get_experiment_ids() if USE_MANUAL_IDS else None,
)


---
## Results

Load and compare any results files after running experiments.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 15 — List and compare results                         ║
# ╚══════════════════════════════════════════════════════════════╝
import json, os


def list_results():
    """Print all available result files."""
    files = sorted(f for f in os.listdir(RESULTS_DIR) if f.endswith('.json'))
    if not files:
        print("No results yet. Run an experiment first.")
        return
    print(f"Results in {RESULTS_DIR}:")
    for fname in files:
        with open(f"{RESULTS_DIR}/{fname}") as f:
            d = json.load(f)
        print(f"  {fname}")
        print(f"    experiment={d['experiment']}  param={d['param_name']}  "
              f"n={d['n_samples']}  ts={d['timestamp']}")


def compare_results(file_a, file_b):
    """Print side-by-side summary of two experiment results."""
    a = json.load(open(file_a))
    b = json.load(open(file_b))
    for label, d in [('A', a), ('B', b)]:
        print(f"\n{label}: {d['experiment']}  ({d['param_name']})")
        for v, s in sorted(d['results'].items(), key=lambda x: float(x[0])):
            total_psst  = s['total_consensus'] + s['total_psst_only']
            agree_pct   = 100 * s['total_consensus'] / max(total_psst, 1)
            rising_pct  = 100 * s['inton_rising']    / max(s['total_consensus'], 1)
            falling_pct = 100 * s['inton_falling']   / max(s['total_consensus'], 1)
            level_pct   = 100 * s['inton_level']     / max(s['total_consensus'], 1)
            print(f"  {v:>8} → agree={agree_pct:5.1f}%  "
                  f"rising={rising_pct:5.1f}%  "
                  f"falling={falling_pct:5.1f}%  "
                  f"level={level_pct:5.1f}%")


# ── Usage ─────────────────────────────────────────────────────────
# list_results()
#
# compare_results(
#     f"{RESULTS_DIR}/alignment_tolerance_20250101_120000.json",
#     f"{RESULTS_DIR}/wav2tobi_threshold_mult_20250101_130000.json"
# )

list_results()


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  COORDINATOR — Run all sweeps in dependency order           ║
# ║                                                              ║
# ║  Assumes:                                                    ║
# ║    1. Models are loaded (Cells 2–4)                         ║
# ║    2. Helpers are defined (Cells 5–9)                       ║
# ║    3. Pool is built (Cell 10)                               ║
# ║    4. Diagnostic has been run (optional but recommended)    ║
# ║                                                              ║
# ║  Runs the joint sweep first, then feeds best (tol, mult)    ║
# ║  into the F0 sweeps so F0 results reflect your actual       ║
# ║  chosen Wav2ToBI settings rather than arbitrary defaults.   ║
# ╚══════════════════════════════════════════════════════════════╝

print("=" * 60)
print("  COORDINATOR: full sweep sequence")
print("=" * 60)

# ── Step 1: Diagnostic (runs fast, informs tolerance range) ────────
print("\n── Step 1: Peak-offset diagnostic ──")
diag_offsets = diagnose_peak_offsets()

# ── Step 2: Joint tolerance × threshold sweep ──────────────────────
print("\n── Step 2: Joint sweep (tolerance × threshold_mult) ──")
results_joint, ids_joint, best_tol, best_mult = run_joint_sweep()
print(f"\n  Locked in: tolerance={best_tol}  threshold_mult={best_mult}")

# ── Step 3: F0 window sweep (using best tol + mult) ────────────────
print("\n── Step 3: F0 window sweep ──")

def apply_f0_window_tuned(raw, psst_cache, audio16, mfa_words, psst_indices, window_s):
    detections  = detect_peaks_from_logits(raw, threshold_mult=best_mult)
    w2t_pairs   = wav2tobi_to_word_boundaries(detections, mfa_words,
                                               tolerance_s=best_tol)
    w2t_indices = [idx for idx, _ in w2t_pairs]
    consensus   = compute_consensus(psst_indices, w2t_pairs)
    counts      = classify_agreement_counts(psst_indices, w2t_indices, len(mfa_words))
    inton = {'rising': 0, 'falling': 0, 'level': 0, 'unclear': 0}
    for idx in consensus:
        label = classify_intonation(audio16, 16000, mfa_words[idx]['end'],
                                     window_s=window_s)
        inton[label] += 1
    return {**counts, **inton}

results_f0_window, ids_f0_window = run_sweep(
    experiment_name = "f0_window_tuned",
    param_name      = "F0_WINDOW_S",
    param_values    = F0_WINDOW_VALUES,
    apply_fn        = apply_f0_window_tuned,
    ids             = ids_joint,   # reuse same samples
)

# Extract best F0 window by highest falling% + rising% (most discriminative)
best_f0_window = max(
    results_f0_window,
    key=lambda v: (results_f0_window[v]['inton_rising'] +
                   results_f0_window[v]['inton_falling']) /
                  max(results_f0_window[v]['total_consensus'], 1)
)
print(f"\n  Locked in: f0_window_s={best_f0_window}")

# ── Step 4: F0 slope sweep (using best tol + mult + f0_window) ─────
print("\n── Step 4: F0 slope sweep ──")

def apply_f0_slope_tuned(raw, psst_cache, audio16, mfa_words, psst_indices, slope_hz):
    detections  = detect_peaks_from_logits(raw, threshold_mult=best_mult)
    w2t_pairs   = wav2tobi_to_word_boundaries(detections, mfa_words,
                                               tolerance_s=best_tol)
    w2t_indices = [idx for idx, _ in w2t_pairs]
    consensus   = compute_consensus(psst_indices, w2t_pairs)
    counts      = classify_agreement_counts(psst_indices, w2t_indices, len(mfa_words))
    inton = {'rising': 0, 'falling': 0, 'level': 0, 'unclear': 0}
    for idx in consensus:
        label = classify_intonation(audio16, 16000, mfa_words[idx]['end'],
                                     window_s=best_f0_window,
                                     slope_thresh=slope_hz)
        inton[label] += 1
    return {**counts, **inton}

results_f0_slope, ids_f0_slope = run_sweep(
    experiment_name = "f0_slope_tuned",
    param_name      = "F0_SLOPE_HZ",
    param_values    = F0_SLOPE_VALUES,
    apply_fn        = apply_f0_slope_tuned,
    ids             = ids_joint,
)

best_f0_slope = max(
    results_f0_slope,
    key=lambda v: (results_f0_slope[v]['inton_rising'] +
                   results_f0_slope[v]['inton_falling']) /
                  max(results_f0_slope[v]['total_consensus'], 1)
)

# ── Final summary ────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  COORDINATOR COMPLETE — Recommended configuration:")
print("=" * 60)
print(f"  ALIGNMENT_TOLERANCE_S    = {best_tol}")
print(f"  W2T_THRESHOLD_MULT       = {best_mult}")
print(f"  F0_WINDOW_S              = {best_f0_window}")
print(f"  F0_SLOPE_HZ              = {best_f0_slope}")
print()
print("  Copy these into Cell 1 of annotation_pipeline.ipynb")
print("  (DEFAULT_* variables) before running your full dataset.")
print("=" * 60)

### Manual annotation of the 50 clips, sees where I agree with the F0 slope most frequently

In [ ]:
def find_critical_items_all_thresholds(sample_ids, thresholds=F0_SLOPE_VALUES):
    """
    For every consensus boundary, classify it at every threshold value.
    Only keep boundaries where at least one threshold disagrees with another
    (i.e. the threshold choice actually matters for this boundary).
    Saves one audio snippet per critical item.
    """
    import soundfile as sf
    os.makedirs(f"{SANDBOX_ROOT}/critical_items", exist_ok=True)
    critical = []

    for sample_id in sample_ids:
        cached = load_psst_cache(sample_id)
        if cached is None:
            continue
        mfa_words = cached['mfa_words']

        result = load_audio_for_sample(sample_id)
        if result is None:
            continue
        audio, sr, _ = result
        audio16 = librosa.resample(audio, orig_sr=sr, target_sr=16000) \
                  if sr != 16000 else audio

        raw          = run_wav2tobi_raw(audio16)
        detections   = detect_peaks_from_logits(raw, threshold_mult=best_mult)
        w2t_pairs    = wav2tobi_to_word_boundaries(detections, mfa_words,
                                                    tolerance_s=best_tol)
        psst_indices = psst_to_word_boundaries(cached['psst_raw'], mfa_words)
        consensus    = compute_consensus(psst_indices, w2t_pairs)

        for idx in consensus:
            word_end = mfa_words[idx]['end']

            # Classify at every threshold
            labels = {
                thresh: classify_intonation(audio16, 16000, word_end,
                                             slope_thresh=thresh)
                for thresh in thresholds
            }

            # Only keep if not all thresholds agree
            if len(set(labels.values())) == 1:
                continue

            # Extract snippet: 600ms before boundary, 400ms after
            start_s = max(0.0, word_end - 0.6)
            end_s   = min(len(audio16) / 16000, word_end + 0.4)
            snippet = audio16[int(start_s * 16000): int(end_s * 16000)]

            fname = f"{sample_id}_w{idx}_{mfa_words[idx]['word']}.wav"
            fpath = f"{SANDBOX_ROOT}/critical_items/{fname}"
            sf.write(fpath, snippet, 16000)

            critical.append({
                'sample_id':  sample_id,
                'word_idx':   idx,
                'word':       mfa_words[idx]['word'],
                'word_end_s': round(word_end, 3),
                'labels':     {str(k): v for k, v in labels.items()},
                'snippet':    fpath,
            })

    print(f"Found {len(critical)} critical items across {len(thresholds)} thresholds")
    return critical

critical_items = find_critical_items_all_thresholds(pool[:N_EXPERIMENT])

In [ ]:
def print_annotation_sheet(critical_items, seed=42):
    """
    Shuffle and print annotation sheet. No labels shown.
    Saves randomised key to annotation_key.json — don't open until done.
    """
    import random
    items = critical_items.copy()
    random.seed(seed)
    random.shuffle(items)

    print("ANNOTATION SHEET")
    print("Listen to each clip. Write: R (rising)  F (falling)  L (level)\n")
    print(f"{'#':<5} {'Clip filename':<50} Your label")
    print("-" * 70)
    for i, item in enumerate(items):
        fname = os.path.basename(item['snippet'])
        print(f"{i+1:<5} {fname:<50} ___")

    key_path = f"{SANDBOX_ROOT}/critical_items/annotation_key.json"
    with open(key_path, 'w') as f:
        json.dump([{'n': i+1, **item} for i, item in enumerate(items)], f, indent=2)

    print(f"\n{len(items)} items total.")
    print(f"Key saved → {key_path}  (don't open until finished annotating)")

print_annotation_sheet(critical_items)

In [ ]:
# This code was written by Gemini ->

import re
import os
from IPython.display import Audio, display, clear_output
import ipywidgets as widgets

# 1. --- CONFIGURATION ---
ROOT_DIR = '/content/drive/MyDrive/Capstone/sandbox/audio'
raw_input = """
1     7190_90543_000008_000003_w16_me.wav                ___
2     5750_100289_000006_000002_w12_period.wav           ___
3     669_129074_000026_000003_w9_course.wav             ___
4     909_131044_000016_000001_w46_body,.wav             ___
5     3526_176651_000012_000000_w38_below..wav           ___
6     7190_90543_000028_000000_w3_necessity.wav          ___
7     3168_173564_000009_000000_w0_"Yes,.wav             ___
8     7190_90543_000008_000003_w38_society,.wav          ___
9     7190_90543_000008_000003_w22_skeleton,.wav         ___
10    5750_100289_000006_000002_w19_materially;.wav      ___
11    669_129061_000004_000000_w23_ready.wav             ___
12    3168_173564_000009_000000_w5_replied.wav           ___
13    6272_70171_000005_000009_w22_continue,.wav         ___
14    2910_131096_000013_000004_w4_wonder,.wav           ___
15    7511_102420_000013_000005_w18_themselves.wav       ___
16    909_131041_000009_000000_w11_States.wav            ___
17    87_121553_000033_000000_w16_vineyard,.wav          ___
18    7190_90542_000099_000000_w17_us,.wav               ___
19    6454_93938_000032_000001_w4_so.wav                 ___
20    87_121553_000119_000000_w6_us.wav                  ___
21    2518_154825_000016_000001_w5_steeper.wav           ___
22    7190_90543_000008_000003_w42_papers.wav            ___
23    3526_175658_000011_000000_w13_opinion;.wav         ___
24    7190_90543_000074_000000_w7_doctor,".wav           ___
25    6454_93938_000032_000001_w12_mail.wav              ___
26    6272_70171_000034_000004_w5_said;.wav              ___
27    5750_100289_000006_000002_w34_service,.wav         ___
28    5789_57158_000040_000001_w7_that.wav               ___
29    6272_70171_000034_000004_w13_Somers.wav            ___
30    909_131041_000009_000000_w7_Magistrate.wav         ___
31    1263_141777_000037_000002_w4_cry.wav               ___
32    2518_154825_000016_000001_w11_slower.wav           ___
33    909_131044_000016_000001_w8_conclusive,.wav        ___
34    2910_131096_000013_000004_w8_building.wav          ___
35    7511_102419_000005_000001_w10_ground,.wav          ___
36    2518_154825_000016_000001_w13_slower,.wav          ___
37    1263_141777_000023_000004_w12_arm,.wav             ___
38    669_129061_000004_000000_w4_sermons,.wav           ___
39    7511_102419_000005_000001_w31_her.wav              ___
40    669_129061_000004_000000_w21_mornings,.wav         ___
41    7190_90542_000099_000000_w20_flames.wav            ___
42    909_131041_000009_000000_w19_system,.wav           ___
43    2836_5355_000073_000000_w3_me-for.wav              ___
44    6454_93938_000032_000001_w6_realized.wav           ___
45    7190_90543_000074_000000_w16_interview,.wav        ___
46    669_129061_000004_000000_w27_table,.wav            ___
47    5750_100289_000006_000002_w25_because.wav          ___
48    3168_173564_000009_000000_w3_did,".wav             ___
49    6415_116629_000043_000000_w19_Blasi,.wav           ___
50    5789_57158_000040_000001_w14_her.wav               ___
51    7190_90543_000008_000003_w4_not,.wav               ___
52    5322_7680_000011_000002_w6_afraid?.wav             ___
53    1263_141777_000023_000004_w27_busy.wav             ___
54    87_121553_000062_000000_w15_Chiana.wav             ___
55    909_131041_000009_000000_w22_consequence,.wav      ___
56    3168_173564_000009_000000_w1_really.wav            ___
57    6272_70171_000005_000009_w29_plate,.wav            ___
58    1263_141777_000025_000000_w0_Again.wav             ___
59    5322_7680_000011_000002_w10_afraid,'.wav           ___
60    1263_141777_000037_000002_w11_now,.wav             ___
61    7511_102419_000015_000001_w16_story.wav            ___
62    3526_176651_000012_000000_w26_gown.wav             ___
63    7190_90542_000099_000000_w12_fire.wav              ___
64    7190_90543_000008_000003_w29_secret,.wav           ___
65    909_131041_000009_000000_w28_censure,.wav          ___
66    6415_116629_000043_000000_w8_Gertrude,.wav         ___
67    5322_7680_000023_000002_w4_ourselves.wav           ___
68    87_121553_000062_000000_w18_heaven.wav             ___
69    7190_90543_000008_000003_w11_question,.wav         ___
"""

# 2. --- PARSING & CLEANING ---
def parse_line(line):
    parts = line.split()
    if len(parts) < 2: return None
    filename = parts[1]

    # Regex captures the base (up to the last 6-digit block) and the suffix context
    match = re.search(r'(.*_\d{6})_(.*)\.wav', filename)
    if match:
        base_path = match.group(1) + ".wav"
        context = match.group(2)
        return {"path": os.path.join(ROOT_DIR, base_path), "context": context}
    return None

# Generate our dataset list
data_list = [parse_line(line) for line in raw_input.strip().split('\n') if parse_line(line)]

# Storage for results
your_labels = {i+1: '' for i in range(len(data_list))}
current_index = 0

# 3. --- UI LOGIC ---
def show_next_file():
    global current_index
    clear_output(wait=True)

    if current_index >= len(data_list):
        print("✅ Labeling Complete!")
        print("\nFinal Dictionary Result:")
        print(your_labels)
        return

    item = data_list[current_index]
    file_path = item["path"]
    context_text = item["context"]

    # Header Info
    print(f"Progress: {current_index + 1} / {len(data_list)}")
    print("-" * 30)
    print(f"Current Word/Context: ** {context_text} **")
    print("-" * 30)

    # Audio Player
    if os.path.exists(file_path):
        display(Audio(file_path, autoplay=True))
    else:
        print(f"❌ File not found: {os.path.basename(file_path)}")

    # Label Buttons
    options = ["falling", "rising", "flat"]
    button_list = []

    for label in options:
        btn = widgets.Button(description=label, layout=widgets.Layout(width='100px', margin='10px'))
        btn.style.button_color = 'lightblue' if label == "flat" else 'lightgreen'
        btn.on_click(on_button_clicked)
        button_list.append(btn)

    display(widgets.HBox(button_list))

def on_button_clicked(b):
    global current_index
    # Save the text label (e.g., "falling")
    your_labels[current_index + 1] = b.description

    current_index += 1
    show_next_file()

# Initial Start
show_next_file()

In [ ]:
F0_SLOPE_VALUES = [2.0, 5.0, 7.5, 10.0, 12.5, 15.0, 20.0, 25.0]
def score_against_all_thresholds(your_labels: dict, thresholds=F0_SLOPE_VALUES):
    """
    your_labels: {item_number (int): 'rising'/'falling'/'level'}
    Computes accuracy of each threshold value against your human labels,
    then prints a ranked table and highlights the best match.
    """
    key_path = f"{SANDBOX_ROOT}/critical_items/annotation_key.json"
    with open(key_path) as f:
        key = json.load(f)

    # Normalise input labels
    label_map = {'r': 'rising', 'f': 'falling', 'l': 'level',
                 'rising': 'rising', 'falling': 'falling', 'level': 'level',
                 'flat': 'level'}
    normed = {n: label_map.get(str(lbl).strip().lower())
              for n, lbl in your_labels.items()}

    judged_items = [item for item in key if normed.get(item['n']) is not None]
    n = len(judged_items)
    if n == 0:
        print("No labelled items found. Check your_labels keys match item numbers.")
        return

    # Tally correct per threshold
    scores = {thresh: 0 for thresh in thresholds}
    confusion = {thresh: {'tp': 0, 'fp': 0} for thresh in thresholds}

    for item in judged_items:
        human = normed[item['n']]
        for thresh in thresholds:
            predicted = item['labels'][str(float(thresh))]
            if predicted == human:
                scores[thresh] += 1

    # Print ranked results
    print(f"\n{'='*55}")
    print(f"  HUMAN vs THRESHOLD ACCURACY  ({n} items judged)")
    print(f"{'='*55}")
    print(f"  {'Threshold':>10}  {'Correct':>8}  {'Accuracy':>9}")
    print(f"  {'-'*10}  {'-'*8}  {'-'*9}")

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    for thresh, correct in ranked:
        marker = " ← best match" if thresh == ranked[0][0] else ""
        print(f"  {thresh:>10}  {correct:>8}  {100*correct/n:>8.1f}%{marker}")

    print(f"{'='*55}")
    print(f"\n  Recommended F0_SLOPE_HZ = {ranked[0][0]}")
    print(f"  (closest match to your human annotations)")

    # Save results
    results_path = f"{SANDBOX_ROOT}/critical_items/human_eval_results.json"
    with open(results_path, 'w') as f:
        json.dump({
            'n_judged':   n,
            'thresholds': thresholds,
            'scores':     {str(k): v for k, v in scores.items()},
            'accuracies': {str(k): round(v/n, 4) for k, v in scores.items()},
            'best_threshold': ranked[0][0],
        }, f, indent=2)
    print(f"\n  Results saved → {results_path}")

# ── Fill this in after annotating ────────────────────────────────────
# Keys are item numbers from the printed sheet, values are your labels.
# Accepts 'rising'/'falling'/'level' or shorthand 'r'/'f'/'l'.

my_labels = {1: 'flat', 2: 'rising', 3: 'falling', 4: 'falling', 5: 'falling',
                  6: 'flat', 7: 'flat', 8: 'flat', 9: 'falling',
                  10: 'falling', 11: 'falling', 12: 'flat', 13: 'falling', 14: 'falling',
                  15: 'flat', 16: 'falling', 17: 'falling', 18: 'falling', 19: 'rising',
                  20: 'falling', 21: 'falling', 22: 'falling', 23: 'falling', 24: 'rising',
                  25: 'rising', 26: 'flat', 27: 'falling', 28: 'flat', 29: 'falling',
                  30: 'rising', 31: 'falling', 32: 'falling', 33: 'rising', 34: 'falling',
                  35: 'falling', 36: 'falling', 37: 'falling', 38: 'falling', 39: 'falling',
                  40: 'falling', 41: 'falling', 42: 'rising', 43: 'rising', 44: 'rising',
                  45: 'rising', 46: 'falling', 47: 'rising', 48: 'falling', 49: 'falling',
                  50: 'flat', 51: 'flat', 52: 'rising', 53: 'falling', 54: 'falling',
                  55: 'falling', 56: 'rising', 57: 'flat', 58: 'rising', 59: 'rising',
                  60: 'rising', 61: 'falling', 62: 'falling', 63: 'falling', 64: 'rising',
                  65: 'falling', 66: 'falling', 67: 'falling', 68: 'falling', 69: 'falling'}

score_against_all_thresholds(my_labels)

2.0 Hz matches human perceptual judgement on ambiguous items the best.

These 69 items are FILTERED to critical items (where different thresholds were in disagreement), meaning these are the toughest examples to annotate. Manual annotation has shown that all 69 examples were devoid of background noise or other interfering factors, meaning F0 is a pretty good indicator as to pitch.

51% is significantly significant (since 33% is random chance), though it's important to understand that F0 slope is still not a perfect indicator as to classify whether boundaries are rising or falling in nature

Going to run this though in case window + slope in tandem gives a different results. Important to note that human annotation had 59% falling, 25% rising, 16% flat - it should most closely resemble this

In [ ]:
F0_WINDOW_VALUES = [0.10, 0.15, 0.20, 0.25, 0.30]
F0_SLOPE_VALUES  = [2.0, 5.0, 7.5, 10.0]

def apply_f0_joint(raw, psst_raw, audio16, mfa_words, psst_indices, params):
    window_s, slope_hz = params
    detections  = detect_peaks_from_logits(raw, threshold_mult=best_mult)
    w2t_pairs   = wav2tobi_to_word_boundaries(detections, mfa_words,
                                               tolerance_s=best_tol)
    w2t_indices = [idx for idx, _ in w2t_pairs]
    counts      = classify_agreement_counts(psst_indices, w2t_indices, len(mfa_words))
    consensus   = compute_consensus(psst_indices, w2t_pairs)
    inton = {'rising': 0, 'falling': 0, 'level': 0, 'unclear': 0}
    for idx in consensus:
        inton[classify_intonation(audio16, 16000, mfa_words[idx]['end'],
                                   window_s=window_s,
                                   slope_thresh=slope_hz)] += 1
    return {**counts, **inton}

import itertools
combos = list(itertools.product(F0_WINDOW_VALUES, F0_SLOPE_VALUES))

results_f0_joint, _ = run_sweep(
    experiment_name = "f0_window_x_slope",
    param_name      = "F0_WINDOW_S x F0_SLOPE_HZ",
    param_values    = combos,
    apply_fn        = apply_f0_joint,
    ids             = ids_joint,
)

In [ ]:
human_dist = {'falling': 0.59, 'rising': 0.25, 'level': 0.16}

print(f"\n{'Window':>8}  {'Slope':>6}  {'Fall%':>6}  {'Rise%':>6}  {'Flat%':>6}  {'MAE':>6}")
print("-" * 50)

rows = []
for params, s in results_f0_joint.items():
    window_s, slope_hz = params
    total = max(s['total_consensus'], 1)
    fall  = s['inton_falling'] / total
    rise  = s['inton_rising']  / total
    flat  = s['inton_level']   / total
    mae   = (abs(fall - human_dist['falling']) +
             abs(rise - human_dist['rising'])  +
             abs(flat - human_dist['level'])) / 3
    rows.append((window_s, slope_hz, fall, rise, flat, mae))

for row in sorted(rows, key=lambda x: x[5]):  # sort by MAE ascending
    window_s, slope_hz, fall, rise, flat, mae = row
    print(f"{window_s:>8}  {slope_hz:>6}  "
          f"{100*fall:>5.1f}%  {100*rise:>5.1f}%  "
          f"{100*flat:>5.1f}%  {mae:>6.3f}")

## Final Configuration

```
ALIGNMENT_TOLERANCE_S = 0.15

W2T_THRESHOLD_MULT    = 1.0

F0_WINDOW_S           = 0.20

F0_SLOPE_HZ           = 2.0
```